In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from lightgbm import LGBMClassifier

RANDOM_STATE_1 = 42
RANDOM_STATE_2 = 2024
N_SPLITS = 5


def mcc_optimal_threshold(y_true, proba, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)
    scores = []
    for t in grid:
        scores.append(matthews_corrcoef(y_true, (proba >= t).astype(int)))
    best_i = int(np.argmax(scores))
    return float(grid[best_i]), float(scores[best_i])


def cv_oof_and_test_proba_lgb(X, y, X_test, seed, n_splits=5):
    params = dict(
        n_estimators=6000,
        learning_rate=0.015,
        num_leaves=63,
        min_child_samples=15,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=seed,
        n_jobs=-1,
    )

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof = np.zeros(len(X), dtype=float)
    test_proba = np.zeros(len(X_test), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = LGBMClassifier(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
        )

        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        test_proba += model.predict_proba(X_test)[:, 1] / n_splits

    return oof, test_proba


def cv_oof_and_test_proba_hgb(X, y, X_test, seed, n_splits=5):
    # HGB doesn’t expose predict_proba unless loss is logistic (classification default ok)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof = np.zeros(len(X), dtype=float)
    test_proba = np.zeros(len(X_test), dtype=float)

    # Parameters that usually work well on numeric engineered features
    base_params = dict(
        max_depth=6,
        learning_rate=0.05,
        max_iter=600,
        min_samples_leaf=30,
        l2_regularization=0.0,
        random_state=seed,
    )

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = HistGradientBoostingClassifier(**base_params)
        model.fit(X_tr, y_tr)

        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        test_proba += model.predict_proba(X_test)[:, 1] / n_splits

    return oof, test_proba


# =====================
# Load data
# =====================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

ID_COL = train.columns[0]
TARGET_COL = "target_class"

test_ids = test[ID_COL].values

X = train.drop(columns=[ID_COL, TARGET_COL])
y = train[TARGET_COL].astype(int)
X_test = test.drop(columns=[ID_COL])

# =====================
# Base models OOF + test probas
# =====================
oof_lgb1, test_lgb1 = cv_oof_and_test_proba_lgb(
    X, y, X_test, seed=RANDOM_STATE_1, n_splits=N_SPLITS
)
oof_lgb2, test_lgb2 = cv_oof_and_test_proba_lgb(
    X, y, X_test, seed=RANDOM_STATE_2, n_splits=N_SPLITS
)
oof_hgb, test_hgb = cv_oof_and_test_proba_hgb(
    X, y, X_test, seed=RANDOM_STATE_1, n_splits=N_SPLITS
)

# Stack features: OOF for meta-training, averaged test probs for meta-inference
X_stack_oof = np.column_stack([oof_lgb1, oof_lgb2, oof_hgb])
X_stack_test = np.column_stack([test_lgb1, test_lgb2, test_hgb])

# =====================
# Meta model (strong regularization to avoid overfit)
# =====================
meta = LogisticRegression(
    penalty="l2",
    C=0.05,  # strong reg; try 0.02 / 0.05 / 0.1 if you want
    solver="lbfgs",
    max_iter=2000,
)
meta.fit(X_stack_oof, y)

meta_oof_proba = meta.predict_proba(X_stack_oof)[:, 1]
t_meta, mcc_meta = mcc_optimal_threshold(y, meta_oof_proba)

meta_test_proba = meta.predict_proba(X_stack_test)[:, 1]
meta_test_pred = (meta_test_proba >= t_meta).astype(int)

print(f"[Meta] Best OOF MCC={mcc_meta:.5f} @ t={t_meta:.3f}")

sub_stack = pd.DataFrame({"ID": test_ids, "target": meta_test_pred})
sub_stack.to_csv("results_stacking.csv", index=False)
print("Saved: results_stacking.csv")

[LightGBM] [Info] Number of positive: 15360, number of negative: 3840
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002245 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5941
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.800000 -> initscore=1.386294
[LightGBM] [Info] Start training from score 1.386294
[LightGBM] [Info] Number of positive: 15360, number of negative: 3840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000751 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5916
[LightGBM] [Info] Number of data points in the train set: 19200, number of used features: 42
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.800000 -> initscore=1.386294
[LightG

/home/matajur/dev/matajur/Woolf/Tier_3/04_career_strategies/test_task_churn_pred/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[Meta] Best OOF MCC=0.89175 @ t=0.635
Saved: results_stacking.csv
